# NeuroScan - Training with IAM Handwriting Database
## Combined Training: IAM (Normal) + Kaggle Dyslexia Dataset

This notebook trains using:
- **IAM Handwriting Database** → Normal handwriting (LOW risk)
- **Kaggle Dyslexia Dataset** → Reversal patterns (HIGH risk)

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload both datasets when prompted
3. Run all cells

In [ ]:
# Cell 1: Install dependencies
!pip install tensorflow opencv-python-headless scikit-learn seaborn -q
print("✅ Dependencies installed!")

In [ ]:
# Cell 2: Check GPU & imports
import tensorflow as tf
import numpy as np
import cv2
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
import random

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU: {gpus[0]}")
else:
    print("⚠️ No GPU found!")

In [ ]:
# Cell 3: Upload datasets
from google.colab import files
import zipfile

os.makedirs('data/iam', exist_ok=True)
os.makedirs('data/dyslexia', exist_ok=True)

print("="*60)
print("UPLOAD YOUR DATASETS")
print("="*60)
print("""
Upload TWO zip files:
1. IAM Handwriting Database (iam_words.zip or similar)
2. Kaggle Dyslexia Dataset (if you have it)

If you only have IAM, that's fine - we'll create synthetic dyslexia samples.
""")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n📦 Processing: {filename}")
    if filename.endswith('.zip'):
        # Determine destination based on filename
        if 'iam' in filename.lower() or 'words' in filename.lower():
            dest = 'data/iam'
        else:
            dest = 'data/dyslexia'
        
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall(dest)
        os.remove(filename)
        print(f"   Extracted to {dest}/")
    elif filename.endswith('.rar'):
        !apt-get install unrar -qq
        # Check if password protected
        if 'dyslexia' in filename.lower():
            !unrar x -pWanAsy321 "{filename}" data/dyslexia/
        else:
            !unrar x "{filename}" data/iam/
        os.remove(filename)

print("\n✅ Upload complete!")

In [ ]:
# Cell 4: Explore uploaded data
print("=== Data Structure ===")
!find data -type d | head -30
print("\n=== Sample files ===")
!find data -name "*.png" | head -10
!find data -name "*.txt" | head -5

In [ ]:
# Cell 5: Find words.txt and parse IAM metadata
import glob

# Find words.txt
words_txt_files = glob.glob('data/**/words.txt', recursive=True)
if not words_txt_files:
    words_txt_files = glob.glob('data/**/*.txt', recursive=True)
    words_txt_files = [f for f in words_txt_files if 'word' in f.lower()]

print(f"Found txt files: {words_txt_files}")

# Parse words.txt
IAM_WORDS = {}  # word_id -> {transcription, status, path}

if words_txt_files:
    words_txt = words_txt_files[0]
    print(f"\nParsing: {words_txt}")
    
    with open(words_txt, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            
            parts = line.split()
            if len(parts) >= 9:
                word_id = parts[0]  # e.g., a01-000u-00-00
                status = parts[1]   # ok or er
                transcription = parts[-1]  # The actual word
                
                # Build expected image path
                # a01-000u-00-00 -> a01/a01-000u/a01-000u-00-00.png
                parts_id = word_id.split('-')
                if len(parts_id) >= 2:
                    folder1 = parts_id[0]  # a01
                    folder2 = f"{parts_id[0]}-{parts_id[1]}"  # a01-000u
                    
                    IAM_WORDS[word_id] = {
                        'transcription': transcription,
                        'status': status,
                        'folder1': folder1,
                        'folder2': folder2,
                    }
    
    print(f"\n✅ Parsed {len(IAM_WORDS)} word entries")
    
    # Show sample
    print("\nSample entries:")
    for i, (k, v) in enumerate(list(IAM_WORDS.items())[:5]):
        print(f"  {k}: '{v['transcription']}' ({v['status']})")
else:
    print("⚠️ words.txt not found - will scan for images directly")

In [ ]:
# Cell 6: Find IAM image base path

# Find where the actual images are
iam_images = glob.glob('data/iam/**/*.png', recursive=True)
print(f"Found {len(iam_images)} IAM images")

if iam_images:
    # Determine base path
    sample_path = iam_images[0]
    print(f"Sample path: {sample_path}")
    
    # Find the 'words' folder or similar
    IAM_BASE = None
    for part in Path(sample_path).parts:
        test_path = str(Path(*Path(sample_path).parts[:Path(sample_path).parts.index(part)+1]))
        if os.path.isdir(test_path):
            # Check if this contains a01, b01, etc.
            subdirs = os.listdir(test_path)
            if any(d.startswith('a0') or d.startswith('b0') for d in subdirs):
                IAM_BASE = test_path
                break
    
    if not IAM_BASE:
        # Just use parent of first image's grandparent
        IAM_BASE = str(Path(iam_images[0]).parent.parent.parent)
    
    print(f"\nIAM base path: {IAM_BASE}")
    print(f"Contents: {os.listdir(IAM_BASE)[:10]}...")
else:
    print("❌ No IAM images found!")
    IAM_BASE = None

In [ ]:
# Cell 7: Configuration
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 50
MAX_IAM_SAMPLES = 5000  # Limit IAM samples to balance with dyslexia data

# For binary classification: Normal vs Dyslexia
CLASS_NAMES = ['Normal', 'Dyslexia']
NUM_CLASSES = 2

print(f"Config: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, MAX_IAM={MAX_IAM_SAMPLES}")

In [ ]:
# Cell 8: Load IAM images (Normal handwriting)

def load_image(path, size=IMG_SIZE):
    """Load and preprocess image."""
    try:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = cv2.resize(img, (size, size))
        img = img.astype(np.float32) / 255.0
        return np.expand_dims(img, axis=-1)
    except:
        return None

# Load IAM images (these are NORMAL handwriting)
print("=== Loading IAM Images (Normal Class) ===")

X_normal = []
loaded_count = 0

# Shuffle and limit
random.seed(42)
shuffled_images = iam_images.copy()
random.shuffle(shuffled_images)

for img_path in shuffled_images:
    if loaded_count >= MAX_IAM_SAMPLES:
        break
    
    img = load_image(img_path)
    if img is not None:
        X_normal.append(img)
        loaded_count += 1
    
    if loaded_count % 1000 == 0:
        print(f"  Loaded {loaded_count} images...")

X_normal = np.array(X_normal)
print(f"\n✅ Loaded {len(X_normal)} NORMAL (IAM) images")

In [ ]:
# Cell 9: Load Kaggle Dyslexia images OR create synthetic

# Check for Kaggle dyslexia data
dyslexia_images = glob.glob('data/dyslexia/**/*.png', recursive=True)
dyslexia_images += glob.glob('data/dyslexia/**/*.jpg', recursive=True)

# Also check for Reversal folder from previous training
reversal_folders = glob.glob('data/**/Reversal', recursive=True)
reversal_folders += glob.glob('data/**/reversal', recursive=True)

for folder in reversal_folders:
    dyslexia_images += glob.glob(f'{folder}/*.png')
    dyslexia_images += glob.glob(f'{folder}/*.jpg')

print(f"Found {len(dyslexia_images)} dyslexia/reversal images")

X_dyslexia = []

if len(dyslexia_images) > 100:
    # Load existing dyslexia images
    print("\n=== Loading Dyslexia Images ===")
    for img_path in dyslexia_images[:MAX_IAM_SAMPLES]:
        img = load_image(img_path)
        if img is not None:
            X_dyslexia.append(img)
    print(f"Loaded {len(X_dyslexia)} dyslexia images")

else:
    # Create SYNTHETIC dyslexia samples by augmenting IAM
    print("\n=== Creating Synthetic Dyslexia Samples ===")
    print("(Applying reversals, distortions to IAM images)")
    
    def create_dyslexia_sample(img):
        """Apply dyslexia-like transformations."""
        h, w = img.shape[:2]
        result = img.copy()
        
        # Random transformations
        transform = random.choice(['flip', 'mirror', 'distort', 'rotate'])
        
        if transform == 'flip':
            # Horizontal flip (letter reversal like b->d)
            result = cv2.flip(result, 1)
        elif transform == 'mirror':
            # Vertical flip (like p->d confusion)
            result = cv2.flip(result, 0)
        elif transform == 'distort':
            # Add perspective distortion
            pts1 = np.float32([[0,0], [w,0], [0,h], [w,h]])
            offset = random.randint(5, 15)
            pts2 = np.float32([[offset,offset], [w-offset,0], [0,h-offset], [w,h]])
            M = cv2.getPerspectiveTransform(pts1, pts2)
            result = cv2.warpPerspective(result, M, (w, h), borderValue=1.0)
        elif transform == 'rotate':
            # Rotate slightly (writing angle issues)
            angle = random.uniform(-15, 15)
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
            result = cv2.warpAffine(result, M, (w, h), borderValue=1.0)
        
        # Add some noise
        noise = np.random.normal(0, 0.05, result.shape)
        result = np.clip(result + noise, 0, 1)
        
        return result.astype(np.float32)
    
    # Create synthetic samples from IAM
    num_synthetic = min(len(X_normal), MAX_IAM_SAMPLES)
    for i in range(num_synthetic):
        synthetic = create_dyslexia_sample(X_normal[i % len(X_normal)])
        X_dyslexia.append(synthetic)
        
        if (i+1) % 1000 == 0:
            print(f"  Created {i+1} synthetic samples...")
    
    print(f"\n✅ Created {len(X_dyslexia)} synthetic DYSLEXIA samples")

X_dyslexia = np.array(X_dyslexia)

In [ ]:
# Cell 10: Combine and balance dataset

# Create labels
y_normal = np.zeros(len(X_normal), dtype=np.int32)    # 0 = Normal
y_dyslexia = np.ones(len(X_dyslexia), dtype=np.int32)  # 1 = Dyslexia

# Balance classes
min_samples = min(len(X_normal), len(X_dyslexia))
print(f"Balancing: {len(X_normal)} normal, {len(X_dyslexia)} dyslexia -> {min_samples} each")

# Random sample to balance
np.random.seed(42)
normal_idx = np.random.choice(len(X_normal), min_samples, replace=False)
dyslexia_idx = np.random.choice(len(X_dyslexia), min_samples, replace=False)

X = np.concatenate([X_normal[normal_idx], X_dyslexia[dyslexia_idx]])
y = np.concatenate([y_normal[normal_idx], y_dyslexia[dyslexia_idx]])

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

print(f"\n✅ Final dataset: {len(X)} samples")
print(f"   Normal: {np.sum(y==0)}, Dyslexia: {np.sum(y==1)}")

In [ ]:
# Cell 11: Visualize samples

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle('Dataset Samples', fontsize=14)

# Normal samples
normal_idx = np.where(y == 0)[0][:6]
for i, idx in enumerate(normal_idx):
    axes[0, i].imshow(X[idx].squeeze(), cmap='gray')
    axes[0, i].set_title('NORMAL', color='green')
    axes[0, i].axis('off')

# Dyslexia samples
dyslexia_idx = np.where(y == 1)[0][:6]
for i, idx in enumerate(dyslexia_idx):
    axes[1, i].imshow(X[idx].squeeze(), cmap='gray')
    axes[1, i].set_title('DYSLEXIA', color='red')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Split data
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# Cell 13: Build model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("✅ Model built!")
model.summary()

In [ ]:
# Cell 14: Data augmentation & callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    fill_mode='constant',
    cval=1.0
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

print("✅ Ready to train!")

In [ ]:
# Cell 15: TRAIN
print("="*60)
print("🚀 TRAINING MODEL")
print("="*60)

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# Cell 16: Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history.history['auc'], label='Train')
axes[2].plot(history.history['val_auc'], label='Val')
axes[2].set_title('AUC')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

In [ ]:
# Cell 17: Evaluate
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('confusion_matrix.png')
plt.show()

test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")
print(f"✅ Test AUC: {test_auc:.4f}")

In [ ]:
# Cell 18: Save model
os.makedirs('trained_models', exist_ok=True)

model.save('trained_models/dyslexia_model.keras')
model.save('trained_models/dyslexia_model.h5')

# Config for binary classification
config = {
    'IMG_SIZE': IMG_SIZE,
    'NUM_CLASSES': 2,
    'CLASS_NAMES': CLASS_NAMES,
    'model_type': 'binary-classification',
    'test_accuracy': float(test_acc),
    'test_auc': float(test_auc),
    'training_data': 'IAM + Synthetic/Kaggle Dyslexia'
}

with open('trained_models/config.json', 'w') as f:
    json.dump(config, f, indent=2)

import shutil
shutil.copy('training_history.png', 'trained_models/')
shutil.copy('confusion_matrix.png', 'trained_models/')

print("\n✅ Model saved!")
for f in os.listdir('trained_models'):
    size = os.path.getsize(f'trained_models/{f}') / 1024
    print(f"  📄 {f} ({size:.1f} KB)")

In [ ]:
# Cell 19: Download
!zip -r neuroscan_iam_model.zip trained_models/

from google.colab import files
files.download('neuroscan_iam_model.zip')

print("""
========================================
🎉 DONE!
========================================

1. Download neuroscan_iam_model.zip
2. Extract to neurascan-python-ai/models/
3. Deploy!
""")

In [ ]:
# Cell 20: Updated integration code for BINARY classification
print("""
# ============================================================
# UPDATE ml_models.py - Binary Classification Version
# ============================================================

# In _analyze_with_cnn function, change the scoring:

def _analyze_with_cnn(image_path: str) -> Dict:
    model, config = load_trained_model()
    if model is None:
        return None
    
    img_size = config.get('IMG_SIZE', 64)
    num_classes = config.get('NUM_CLASSES', 2)
    
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    predictions = []
    for c in contours[:50]:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < 100:
            continue
        letter = cv2.resize(gray[y:y+h, x:x+w], (img_size, img_size))
        letter = np.expand_dims(np.expand_dims(letter.astype(np.float32)/255.0, -1), 0)
        try:
            pred = model.predict(letter, verbose=0)[0]
            predictions.append(pred)
        except:
            continue
    
    if predictions:
        avg_prob = np.mean(predictions)  # Binary: probability of dyslexia
    else:
        resized = cv2.resize(gray, (img_size, img_size))
        resized = np.expand_dims(np.expand_dims(resized.astype(np.float32)/255.0, -1), 0)
        avg_prob = model.predict(resized, verbose=0)[0][0]
    
    # Binary: probability is directly the dyslexia score
    dyslexia_score = float(avg_prob) * 100
    
    if dyslexia_score >= 70:
        confidence = 'HIGH'
        recommendation = 'Strong indicators detected. Professional evaluation recommended.'
    elif dyslexia_score >= 40:
        confidence = 'MODERATE'
        recommendation = 'Some indicators detected. Further assessment suggested.'
    else:
        confidence = 'LOW'
        recommendation = 'Writing patterns within typical range.'
    
    return {
        'dyslexia_score': round(dyslexia_score, 2),
        'confidence': confidence,
        'recommendation': recommendation,
        'predicted_class': 'Dyslexia' if dyslexia_score >= 50 else 'Normal',
        'probability': f'{avg_prob*100:.1f}%',
        'letters_analyzed': len(predictions) if predictions else 1,
    }
""")